# Linear discriminant analysis for concrete strength classification

In previous notebooks, we used a dataset containing concrete mix design variables together with the corresponding compressive strength values.

In this notebook, rather than predicting the compressive strength from a regression perspective, we discretize the target variable into three strength classes:
- Low strength: $f_c < 25\ \text{MPa}$
- Medium strength: $25 \leq f_c \leq 40\ \text{MPa}$
- High strength: $f_c > 40\ \text{MPa}$
  
We will then reformulate the original regression problem as a multiclass classification problem, and use linear discriminant analysis (LDA) to investigate whether Gaussian class models can separate the three concrete classes from the mix design predictors.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

### Load data

The dataset contains:
- Input variables describing the concrete composition
- One target variable corresponding to the concrete compressive strength

In [2]:
# Load data from Excel
file_path = "../Data/Concrete/Concrete.xls"
df = pd.read_excel(file_path)
# The dataset is stored in Excel format (.xls).
# Pandas requires the 'xlrd' package to read this format.
# If you get an error, install it via: pip install xlrd.

# Display the first rows of the dataset
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../Data/Concrete/Concrete.xls'

In [ ]:
# In this dataset, the last column is the concrete compressive strength target
X = df.iloc[1:, :-1].values # All columns except last → input features
y = df.iloc[1:, -1].values # Last column → target

N, d = X.shape

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print(f"Number of samples: {N}")
print(f"Number of input features: {d}")

### Convert compressive strength into strength classes

In [ ]:
# Strength intervals (MPa)
bins = [0, 25, 40, np.inf] ############

class_names = ["Low strength", "Medium strength", "High strength",]

y_class = pd.cut(y,bins=bins,labels=class_names)

plt.figure(figsize=(6,4))
y_class.value_counts().sort_index().plot.bar()

plt.ylabel("Number of samples")

plt.show()

### Split and normalize the data

LDA estimates covariance matrices, variables should therefore have comparable scales.

- Split using 80% train and 20% test
- Compute normalization statistics from training data
- Consistently apply normalization to the test data

In [ ]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=4, stratify=y_class)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train the LDA model

LDA provides a probabilistic classifier that assumes Gaussian-distributed features with a common covariance matrix across the classes.

In [ ]:
lda = LinearDiscriminantAnalysis()

lda.fit(X_train_scaled, y_train)

### Prediction and visualization

The performance of a classifier can be evaluated by comparing its predictions with the ground-truth labels. A convenient way to summarize the classification results is with a confusion matrix, which reports the number of correctly and incorrectly classified samples.

The confusion matrix is often complemented by the accuracy metric, which measures the proportion of correctly classified samples.

In [ ]:
y_pred = lda.predict(X_test_scaled)
accuracy = accuracy_score(y_test,y_pred)

cm = confusion_matrix(y_test, y_pred, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low strength", "Medium strength", "High strength"])
disp.plot(cmap='Blues', xticks_rotation=25)
disp.im_.set_clim(0, 1)
plt.title(f"Accuracy = {accuracy:.2%}")
plt.show()

### Posterior probabilities

We can inspect the posterior probabilities that LDA assigns to each class for every testing instance. These probabilities quantify the model's confidence that a given sample belongs to each of the three concrete strength classes.

In [ ]:
probabilities = lda.predict_proba(X_test_scaled)

pd.DataFrame(probabilities[:10], # Display only the first ten instances
             columns=class_names)